# Landmark SLAM with GTSAM

A robot observes a cube (8 landmarks) from 5 poses. Noisy ICP gives odometry; noisy depth gives landmark observations. Joint optimisation refines both.

Tested with `gtsam==4.3.0`, `scipy`, `open3d>=0.17`. Helpers live in `landmarkSlam.py`.

Note: Open3D visualisation needs a display. In headless/Colab runs keep `SHOW_3D = False`.


In [ ]:
import numpy as np

from landmarkSlam import (
    addNoiseCubes,
    getFrames,
    getLocalCubes,
    getRelativeEdge,
    getVertices,
    icpTransformations,
    optimize,
    readG2o,
    readLandmarks,
    writeG2o,
)

SHOW_3D = False  # set True locally to open Open3D windows


## 1. Ground truth: cube + 5 robot poses


In [ ]:
vertices, points = getVertices()
frames, poses = getFrames()
print(f"{len(points)} landmarks, {len(poses)} poses")
print("poses (x, y, z, yaw_deg):")
for p in poses:
    print(" ", p)
if SHOW_3D:
    from landmarkSlam import visualizeData
    visualizeData(vertices, frames)


## 2. Local observations + noise

Each pose observes the cube in its own frame. Two noise levels: high (`1.8`) for the clouds fed to ICP (odometry), low (`0.15`) for the landmark measurements. If both had the same noise, optimisation could barely improve anything.


In [ ]:
gt_cubes = getLocalCubes(points, poses)
noisy_high = addNoiseCubes(gt_cubes, noise=1.8, seed=42)
noisy_low = addNoiseCubes(gt_cubes, noise=0.15, seed=1)
print(f"cubes: {gt_cubes.shape} (poses × landmarks × xyz)")


## 3. Odometry from ICP

Point-to-point ICP with known correspondences (teaching shortcut for data association). Uses `open3d.pipelines.registration` (the old `o3d.registration` was removed) and `Rotation.as_matrix()` (the old `as_dcm` was removed from SciPy).


In [ ]:
trans = icpTransformations(noisy_high, show=SHOW_3D)
print(f"{len(trans)} relative motions, e.g. T1_2:\n{np.round(trans[0], 3)}")
if SHOW_3D:
    from landmarkSlam import registerCubes
    registerCubes(trans, noisy_low, show=True)


## 4. Build the g2o graph

- `VERTEX_SE3:QUAT` represents the 5 robot poses; `VERTEX_TRACKXYZ` represents the 8 cube landmarks (first frame → world).
- `EDGE_SE3:QUAT` odometry uses a 6×6 information matrix. Landmark observations use `EDGE_SE3_TRACKXYZ` with a 3×3 point information matrix (`160 …`), so landmarks are not treated as poses with unobservable orientation.
- `PARAMS_SE3OFFSET 0` declares the zero sensor offset required by `EDGE_SE3_TRACKXYZ`.
- GTSAM reads `EDGE_SE3_TRACKXYZ` as a 3D bearing-range factor, so each landmark has only three observable coordinates.
- `FIX 1` is written for g2o tools; `optimize()` also adds an explicit GTSAM prior because `gtsam.readG2o()` does not convert `FIX` into a factor.


In [ ]:
noise_path = writeG2o(trans, noisy_low, path="noise.g2o")
print(f"wrote {noise_path}")


## 5. Optimise with GTSAM


In [ ]:
opt_path = optimize(noise_path, "opt_gtsam.g2o")
print(f"wrote {opt_path}")


## 6. Read back + check

The graph residual is non-zero because the measurements are noisy; the ground-truth RMSEs below show whether poses and landmarks actually improved.


In [ ]:
opt_poses = readG2o(opt_path)
opt_landmarks = readLandmarks(opt_path)
opt_edges = getRelativeEdge(opt_poses)
true_positions = np.asarray([[p[0], p[1], p[2]] for p in poses])
true_landmarks = np.asarray(points, dtype=float)
pose_rmse = np.sqrt(np.mean(np.sum((opt_poses[:, :3, 3] - true_positions) ** 2, axis=1)))
landmark_rmse = np.sqrt(np.mean(np.sum((opt_landmarks - true_landmarks) ** 2, axis=1)))
print(f"optimised poses: {opt_poses.shape}, landmarks: {opt_landmarks.shape}")
print(f"robot position RMSE: {pose_rmse:.3f} m")
print(f"landmark position RMSE: {landmark_rmse:.3f} m")
if SHOW_3D:
    from landmarkSlam import registerCubes
    registerCubes(opt_edges, noisy_low, show=True)
